# 02 · QC, trimming and kallisto: decisions and results
**Goal:** check the reads, remove sequencing artefacts, count reads per transcript. **Input:** results of the heavy steps (notebook 01). **Output:** `results/kallisto/`, `results/qc/` (small fastp JSON files are in git). **Kernel:** Python 3 · **Time:** minutes.

The heavy steps run on Euler; this notebook shows their results and records the trimming decision. Set `RUN_LOCALLY = True` below to run them on the laptop instead (hours, ~70 GB disk; every step skips finished work).

| Step | Tool | What it does |
|---|---|---|
| 1 Convert | `fasterq-dump` | `.sra` → FASTQ (reads with a quality per base, mate 1 and 2; deleted after use) |
| 2 Check | FastQC, MultiQC | per-base quality, adapters, duplication per sample; one combined report |
| 3 Clean | fastp | cuts adapters, poly-G tails and low-quality ends; drops low-quality or short reads |
| 4 Count | kallisto | pseudoalignment: the transcripts each read pair is compatible with, reads per transcript (transcript → gene in notebook 03; [docs/pipeline.md](../docs/pipeline.md)) |

Terms: [docs/glossary.md](../docs/glossary.md).

In [1]:
# helpers.py: pipeline functions; runs = sample table
import sys; sys.path.insert(0, '../scripts')
import helpers as h
from IPython.display import display
runs = h.load_runs()
RUN_LOCALLY = False   # False (default): heavy steps run on Euler (notebook 01); True: run them on this laptop

## 1 · Reference and index (local only, ~2 min)
Reads are matched to the transcript sequences of **GENCODE M39** (GRCm39). kallisto first builds an **index** of them (on Euler: `euler/02b_kallisto_index.sh`).

In [2]:
# local only: download the reference files (skipped if present) and build the kallisto index
if RUN_LOCALLY:
    h.download_reference()
    h.build_index()

## 2 · Quality control of all reads
FastQC runs on **all** reads of every run (the `qc` stage on Euler). Its verdicts come from fixed thresholds (`limits.txt`); some WARNs are normal for RNA-seq.

| Check | WARN when | FAIL when | Expected |
|---|---|---|---|
| Per base sequence quality | lower quartile < 10 or median < 25 | < 5 / < 20 | PASS |
| Per sequence quality scores | most common mean quality < 27 | < 20 | PASS |
| **Adapter Content** | > 5 % of reads | > 10 % | PASS, else trim |
| Overrepresented sequences | one sequence > 0.1 % of reads | > 1 % | WARN common (Smart-seq2 primers) |
| Sequence Duplication Levels | < 70 % of reads remain after dedup | < 50 % | WARN normal (highly expressed genes) |

**Result:** base quality PASS in all 54 read files; **adapter content FAIL in all 54** (17–34 % of reads run into Nextera transposase sequence, because fragments are shorter than the 150-base read); overrepresented sequences and duplication WARN, as expected.

In [3]:
# FastQC on every downloaded run (local only), then one combined MultiQC report; the table shows the verdicts of whatever reports exist (from Euler or local)
if RUN_LOCALLY:
    for srr in [s for s in runs.index if h.has_sra(s)]:
        h.fastqc(srr)
    print('report:', h.multiqc())   # open this HTML: all samples side by side
h.fastqc_summary(list(runs.index))

,Per base sequence quality,Per sequence quality scores,Adapter Content,Overrepresented sequences,Sequence Duplication Levels
SRR21355401_1,PASS,PASS,FAIL,WARN,WARN
SRR21355401_2,PASS,PASS,FAIL,WARN,WARN
SRR21355396_1,PASS,PASS,FAIL,WARN,WARN
SRR21355396_2,PASS,PASS,FAIL,WARN,WARN
SRR21355392_1,PASS,PASS,FAIL,WARN,WARN
SRR21355392_2,PASS,PASS,FAIL,WARN,WARN
SRR21355388_1,PASS,PASS,FAIL,WARN,WARN
SRR21355388_2,PASS,PASS,FAIL,WARN,WARN
SRR21355398_1,PASS,PASS,FAIL,WARN,WARN
SRR21355398_2,PASS,PASS,FAIL,WARN,WARN


## 3 · Trim or not?
Adapters fail in **every** sample, so **all** runs are trimmed with the same settings (trimming only some would create a technical difference between groups). fastp detects the adapter from the read pair, trims poly-G tails, keeps reads ≥ 20 bases and drops low-quality reads (about 4–6 % of pairs). Duplicates are **not** removed (see glossary).

In [4]:
# decision recorded for all later cells; the guard in step 5 raises if any run was processed differently
TRIM = True   # FastQC on all 27 runs: Nextera adapter FAIL (17-34 % of reads) + poly-G in every sample; quality PASS

### Check: adapter and poly-G (SRR21355398, first 600,000 read pairs)
**Is it the adapter?** Yes.
- The 12 bases `CTGTCTCTTATA` start the Nextera adapter (expected for a Tn5 library) and occur in 27–30 % of reads, vs ≈ 5 expected by chance.
- The bases after them match the read's *own* mate's adapter in 96–99 % of cases, the other mate's in 0 %.
- After trimming, FastQC adapter content drops from 25–34 % to 0.0 % (two runs).

**Poly-G** (≥ 12 G in a row): the NovaSeq reads a cluster with no light as G. After a short insert (~49 bases) a read runs through the adapter (34), a sample barcode (8) and more library construct (~33), then out of template; the remaining ~24 cycles are dark:
```
read    [ insert ][ adapter ][ barcode ][ rest of construct ][ no template left: dark -> GGGGGGG ]
bases      ~49         34          8            ~33                  ~24
```
Only 4–6 % of reads have a poly-G run (27–30 % reach the adapter). The G's have high quality (mean Q32), so quality trimming misses them. Adapter cutting alone removes them in 91–93 % of these reads, fastp's filters drop 5–6 %, and 1.5–3 % keep a G run (about half real G-rich sequence). `--trim_poly_g` catches the rest, so **it is on for all 27 runs**. Without it (one run tested) pseudoalignment was identical (65.20 %) and counts changed by 0.05 %, but a few G-rich genes (Gm19220, Gm26822, Grip2) were inflated by residual poly-G reads (`results/qc/polyG_sensitivity_SRR21355398.csv`); none is differentially expressed.

In [5]:
# per mate: adapter evidence and the fate of poly-G reads (needs the .sra on this machine; re-runs fastq-dump and fastp on 600k pairs, ~1 min)
if h.has_sra('SRR21355398'):
    display(h.adapter_check('SRR21355398').T)
else:
    print('SRR21355398.sra is not on this machine (notebook 00 downloads it); the results are described in the text above')

mate,1,2
reads,600000.0,600000.0
adapter_start_%,27.2,29.6
expected_by_chance_reads,5.0,5.0
continues_own_adapter_%,95.8,98.5
matches_other_mate_adapter_%,0.0,0.0
polyG_reads_%,6.2,4.3
polyG_after_adapter_%,78.8,88.0
polyG_dropped_%,5.2,5.9
polyG_removed_%,93.3,91.1
polyG_remaining_%,1.5,3.0


## 4 · Test on one run (local only)
The smallest run, SRR21355398, before committing to all 27.

In [6]:
# local only: convert -> trim -> kallisto for one run; then the summary table of all runs that have results
if RUN_LOCALLY:
    h.process('SRR21355398', trim_reads=TRIM)
h.qc_summary(runs)

,pairs_M,aligned_%,q30_%,dup_%,group
SRR21355401,14.9,65.8,89.8,7.6,aging 3mo
SRR21355396,25.7,67.6,87.4,6.9,aging 3mo
SRR21355392,16.1,67.6,88.2,6.6,aging 3mo
SRR21355388,16.9,67.9,88.5,7.2,aging 3mo
SRR21355398,6.4,68.1,86.5,3.3,aging 18mo
SRR21355394,7.8,67.9,86.2,2.9,aging 18mo
SRR21355390,28.2,65.1,88.7,8.0,aging 18mo
SRR21355386,9.3,68.1,86.1,3.3,aging 18mo
SRR21355397,14.2,67.0,88.5,5.6,aging 21mo
SRR21355393,9.8,70.1,87.7,3.8,aging 21mo


Expect a **pseudoaligned** share of 60–80 % of read pairs for mouse mRNA (here 65–72 %); a much lower value would point to a wrong reference or a library problem. **Effect of trimming** (five runs first quantified untrimmed, early vM25 reference): +2.1–2.7 points.

## 5 · All runs (local only)
Skips finished runs. A guard raises an error if runs were processed with different trimming settings.

In [7]:
# process every downloaded run (local only); the guard checks that all runs share the same trimming
print('TRIM =', TRIM)   # must be True: re-run the TRIM cell if the kernel was restarted
h.check_consistent(runs, TRIM)
if RUN_LOCALLY:
    for srr in runs.index:
        if h.has_sra(srr):
            h.process(srr, trim_reads=TRIM)
    h.clean_tmp()
    h.check_consistent(runs, TRIM)

TRIM = True
consistent: all quantified runs used trimmed=True


## 6 · Summary
`pairs_M` = million read pairs; `aligned_%` = pseudoaligned share; `q30_%` = bases with error rate ≤ 1 in 1,000; `dup_%` = fastp's duplicate estimate (3–8 %); `trimmed` should be `True` for all runs.

**Why only 65–72 % `aligned_%`?** kallisto matches spliced transcripts only; reads from introns, unspliced RNA or unannotated sequence are not counted and play no part later. STAR maps 77–89 % of pairs uniquely but counts only 47–59 % in annotated genes. The share is similar in all runs, so none is flagged.

In [8]:
# per-run QC numbers from kallisto and fastp
h.qc_summary(runs)

,pairs_M,aligned_%,q30_%,dup_%,group
SRR21355401,14.9,65.8,89.8,7.6,aging 3mo
SRR21355396,25.7,67.6,87.4,6.9,aging 3mo
SRR21355392,16.1,67.6,88.2,6.6,aging 3mo
SRR21355388,16.9,67.9,88.5,7.2,aging 3mo
SRR21355398,6.4,68.1,86.5,3.3,aging 18mo
SRR21355394,7.8,67.9,86.2,2.9,aging 18mo
SRR21355390,28.2,65.1,88.7,8.0,aging 18mo
SRR21355386,9.3,68.1,86.1,3.3,aging 18mo
SRR21355397,14.2,67.0,88.5,5.6,aging 21mo
SRR21355393,9.8,70.1,87.7,3.8,aging 21mo


In [9]:
# downloaded / quantified / trimmed per run
h.status(runs)

,group,sra_GB,quantified,trimmed
SRR,,,,
SRR21355401,aging 3mo,NaN,True,True
SRR21355396,aging 3mo,NaN,True,True
SRR21355392,aging 3mo,NaN,True,True
SRR21355388,aging 3mo,NaN,True,True
SRR21355398,aging 18mo,0.78,True,True
SRR21355394,aging 18mo,NaN,True,True
SRR21355390,aging 18mo,NaN,True,True
SRR21355386,aging 18mo,NaN,True,True
SRR21355397,aging 21mo,NaN,True,True
